In [ ]:
library(dplyr)
library(tidyverse)
library(ConsensusClusterPlus)
library(caret)
library(ggplot2)
library(clusterProfiler)
library(reshape2)
library(plyr)
library(pheatmap)
library(patchwork)
library(Seurat)
library(paletteer)
library(tidyverse)
library(ggpubr) 
library(ggsci)
library(cluster)
library(FCPS)
library(clustree)
library(ComplexHeatmap)
library(circlize)
source("./Function/3.2.k_means.R") 
source("./Function/3.1.hclust.R") 
source("./Function/3.3.ConstructCAM3.0.R") 
source("./Function/3.4.Num_Clusters.R")
source("./Function/3.4.2.PAC.R")
source("./Function/3.4.3.Silhouette.R")
source("./Function/3.4.1.CopheneticCorCoef.R")
source("./Function/3.4..4.AnalyzeICL.R")

input:
> (1) seuratObj: Seurat object storing processed single-cell expression profiles
> 
> (2) signatureList: List of signature sets for base clustering

In [ ]:
# 1. Base Clustering
####
# Extract expression matrix
erMatrix<-as.data.frame(seuratObj@assays$SCT@data)
# Perform base clustering to identify the optimal clustering results for both
results.CAM <- lapply(signatureList, function(x){
  ErMatrix <- erMatrix[unlist(x),]%>%na.omit()
  kmean.CAM<-k_means(ErMatrix,seed = 1234,nc.min = 2,nc.max = 8)
  hclust.CAM<-hclustfunc(ErMatrix,nc.min = 2,nc.max = 8,seed = 1234,nc.method = "ward.D2")
  results.CAM<-append(kmean.CAM,hclust.CAM) 
})

In [ ]:
# 2 Combine Base Clustering Results and Construct Consensus Matrix
####
Results.CAM <- lapply(results.CAM, function(x){
  CAM1<-as.data.frame(ConstructCAM(x$kmeans_clus))
  CAM2<-as.data.frame(ConstructCAM(x$hclust_clus))
  CAM<-(CAM1+CAM2)/2 
})
CAM <- matrix(0, ncol(Results.CAM[[1]]), ncol(Results.CAM[[1]]))
for(i in 1:length(Results.CAM)){
  CAM <- CAM + Results.CAM[[i]]
}
CAM<-CAM/length(Results.CAM)

In [ ]:
# 3. Perform Consensus Clustering on the Consensus Matrix
####
res_CCP = ConsensusClusterPlus(as.matrix(CAM),maxK=8,reps=1000,pItem=0.8,pFeature=1,seed = 123,
                               distance="euclidean",clusterAlg="hc", plot=NULL,verbose = FALSE,
                               title = plotDir) 

# Automatically determine the optimal number of clusters from the consensus clustering results
plotDir = paste0(outDir,p,"/")
candidateK<-NULL
optimalK<-Num_Clusters(res_CCP,  maxK = 8, minClusterNum = 2,outDir = plotDir)
optimalK<-optimalK[1] # Select the first one if there are multiple optimal clusters
# Or manually specify the number of clusters
# optimalK<-3
Fina_Clus = res_CCP[[optimalK]][["consensusClass"]]